# 2회차 · 기울여서 원하는 곳으로

오늘 끝에는 내 코드로 드론이 바람 속에서 5 cm 안에 머물고, 웨이포인트 다섯 개를 돌고, 원을 따라간다.
그리고 날고 있는 드론의 목표, 게인, 바람을 직접 바꿔 가며 어떻게 움직이는지 본다.

**진행 방식**: 1회차와 같다. `✍️` 셀에서 같이 코드를 치고 `Shift+Enter`로 바로 그래프를 본다. 빈칸은 `___`.
오늘 만드는 부품 셋(①②③)은 드론 칩 안에서 1초에 100번 돈다. 3회차부터는 여섯 대 모두의 칩에서 돈다.
표가 나오는 셀은 **실행하기 전에 답을 먼저 맞혀 보자.**

| 단계 | 내용 |
|---|---|
| 0 | 준비 + numpy 몸풀기 |
| 1 | 기울여야 옆으로 간다 |
| 2 | ① 위치 제어기: 위치 오차 → 원하는 가속도 |
| 3 | ② 가속도 → 추력과 자세 (그림으로 먼저) |
| 4 | ③ 기울기 제한 |
| 5 | 첫 비행: 바람 속 호버 |
| 6 | **실시간 비행: 날면서 바꿔 보기** |
| 7 | ④ 웨이포인트: 링 코스 |
| 8 | 원 궤적과 앞먹임 (같이 보기) |
| 9 | 미션: 비행 시험 세 가지 |
| 집에서 | 칼만 필터 ⑤⑥⑦ (선택) |

## 0. 준비

1회차에 같이 만든 PID(`PID`)는 배경 코드가 불러 두었다. 오늘은 이 PID를 **x, y, z 세 축에 한꺼번에** 쓴다.
오차로 숫자 대신 길이 3짜리 배열을 넣으면 축마다 따로 계산된다.

In [ ]:
from background import *

warmup()
pid = PID(kp=6.0)
pid.update(np.array([1.0, -0.5, 0.0]), 0.01)       # 세 축의 오차를 한 번에

**numpy 몸풀기**: 오늘 빈칸에 나오는 문법은 아래가 전부다. 실행해 보고, 숫자를 바꿔서 한 번 더 실행해 보자.

In [ ]:
a = np.array([3.0, 4.0, 0.0])          # x, y, z 세 칸짜리 배열 (위치, 속도, 가속도가 다 이 모양)
b = np.array([1.0, 0.0, 2.0])
print("칸끼리 빼기     a - b       =", a - b)
print("모든 칸에 곱하기 2 * a      =", 2 * a)
print("앞 두 칸(수평)  a[:2]       =", a[:2])
print("길이            norm(a)     =", np.linalg.norm(a), "  # √(3² + 4² + 0²)")
print("두 칸의 길이    hypot(3, 4) =", np.hypot(3, 4))
c = a.copy(); c[:2] *= 0.5             # 앞 두 칸만 절반으로 (4단계에서 쓴다)
print("앞 두 칸만 줄이기          =", c)
st = {"p": a, "v": b}                  # 이름표로 꺼내 쓰는 묶음 (딕셔너리)
print('이름표로 꺼내기 st["p"]     =', st["p"])

## 1. 기울여야 옆으로 간다

드론의 추력은 늘 기체 위쪽으로만 나온다. 옆으로 가려면 **기울여서 추력의 일부를 옆으로 돌린다.**
킥보드를 탈 때 몸을 기울이는 것과 같다.

높이를 지키려면 추력의 위쪽 몫이 무게와 같아야 한다: $T\cos\theta = mg$.
그때 옆쪽 몫은 $T\sin\theta = mg\tan\theta$ 이고, 질량으로 나누면 옆 가속도

$$a_x = g\tan\theta$$

✍️ 기울기 θ [rad]에서 생기는 옆 가속도를 돌려주자. (`np.tan`)

In [ ]:
def sideways_acc(theta, g=G):
    return ___  # ✍️


tilt_test(sideways_acc)

**보고 생각하기**
- 처음에는 실선이 점선(내 식)을 따라 속도가 붙는다. 기울기가 클수록 빨리 붙는다.
- 빨라질수록 실선이 점선 아래로 처진다. 시뮬레이터는 **공기 저항**까지 계산하기 때문이다.
  식만 믿고 기울이면 원하는 만큼 못 간다. 그래서 위치를 보면서 계속 고치는 피드백이 필요하다 (1회차와 같은 이야기).

그래서 칩은 두 단계로 나뉜다.
- **바깥 루프(팀장)**: 어디로, 얼마나 급하게 갈지 → 원하는 **가속도**를 정한다. 오늘 ①
- **안쪽 루프(운전기사)**: 그 가속도를 내려면 얼마나 기울일지 → **추력과 자세**로 바꾼다. 오늘 ②③
- 그 자세를 실제로 만드는 자세 제어기는 미리 짜 두었다 (1초에 500번).

## 2. ① 위치 제어기: 위치 오차 → 원하는 가속도

1회차 PID에 두 가지를 더한다.

$$\mathbf a = \underbrace{K_p\,\mathbf e_p + K_i\!\int\!\mathbf e_p\,dt}_{\text{1회차 PID (P, I)}} + \underbrace{K_d\,\mathbf e_v}_{\text{브레이크}} + \underbrace{\mathbf a_{ff}}_{\text{앞먹임}}$$

- $\mathbf e_p$ = 목표 위치 − 지금 위치, $\mathbf e_v$ = 목표 속도 − 지금 속도
- **D를 속도 오차로**: 목표가 갑자기 바뀌어도 미분이 튀지 않는다.
- **앞먹임** $\mathbf a_{ff}$: 궤적이 "이 순간 이만큼 가속해야 해"라고 미리 알려 주면 그대로 더한다 (8단계에서 효과를 본다).

`st`는 지금 상태, `sp`는 목표(setpoint)다. 둘 다 `st["p"]`, `sp["v"]`처럼 꺼내 쓰고, 값은 길이 3짜리 배열이다.

**실행 전에 맞혀 보기** (`kp = 6`, `kd = 4`, `ki = 0`): 목표가 +x로 1 m 떨어져 있으면 a는? 제자리인데 +y로 1 m/s 움직이고 있으면?

✍️ 오차 두 개와 출력을 채운다.

In [ ]:
class PositionController:
    def __init__(self, kp=6.0, kd=4.0, ki=2.0, i_limit=0.4, i_band=0.3):
        self.pid = PID(kp, ki, 0.0, i_limit, i_band)     # 위치 오차에 P와 I (1회차 PID)
        self.kd = kd                                     # D는 속도 오차에 곱한다

    def reset(self):
        self.pid.reset()

    def acceleration(self, st, sp, dt):
        e_p = ___  # ✍️
        e_v = ___  # ✍️
        return ___  # ✍️


check_acceleration(PositionController)

두 번째 줄: 제자리인데 +y로 움직이고 있으면, 반대쪽(−y)으로 브레이크를 건다.

## 3. ② 가속도 → 추력과 자세

엘리베이터 바닥은 버티는 몫($g$)에 가속하는 몫($a$)을 더해 민다. 드론도 같다. 내야 할 힘은

$$\mathbf f = m(\mathbf a + g\,\hat z)$$

이 힘 쪽으로 기체의 위쪽(추력이 나오는 쪽)을 돌리면 된다. 코드보다 그림을 먼저 보자.

In [ ]:
force_picture(acc=[5.0, 3.0, 0.0])          # 원하는 가속도 [m/s²]. 숫자를 바꿔서 다시 실행해 보자

**보고 생각하기**
- 파란 화살표 f가 추력이 나와야 할 방향이다. 기체(검은 막대)는 그 화살표에 수직으로 눕는다.
- 옆에서 보면 앞으로 숙이는 각 **pitch = atan2(f_x, f_z)**: 위쪽 몫에 비해 앞쪽 몫이 얼마나 큰가.
- 뒤에서 보면 옆으로 기우는 각 **roll = atan2(−f_y, √(f_x² + f_z²))**. 앞으로 숙인 다음의 위쪽 몫이 √(f_x² + f_z²)다.
  왼쪽(+y)으로 가려면 왼쪽을 내리는데, 오른손 법칙으로 그게 roll − 라서 −f_y가 들어간다.
- 추력 **T = |f|**: 기울수록 위쪽 몫을 지키느라 더 세게 밀어야 한다.
- `acc`를 `[9.81, 0, 0]`, `[0, 5, 0]`, `[0, 0, 3]`으로 바꿔 보자. 각도와 추력이 어떻게 변하나?

이제 코드로 옮긴다. 드론이 다른 쪽(yaw)을 보고 있으면 먼저 −yaw만큼 돌려서 기체 기준(앞, 왼쪽)으로 읽어야 하는데,
**그 두 줄은 채워 두었다.** yaw가 0이면 `fx = f[0]`, `fy = f[1]` 그대로라서 위 그림과 똑같다.

**실행 전에 맞혀 보기**: +x로 g만큼 가속하라고 하면 pitch는 몇 도이고 추력은 호버의 몇 배일까?
앞으로 20 m/s²(중력의 두 배쯤)를 달라고 하면 몇 도 기울까? 약 30°, 45°, 64°, 90°?

✍️ 힘, pitch, roll, 추력 네 줄을 채운다. (`np.arctan2`, `np.hypot`, `np.linalg.norm`)

In [ ]:
def acc_to_thrust_attitude(acc, yaw, mass):
    f = ___  # ✍️ m·(a + g ẑ)
    c, s = np.cos(yaw), np.sin(yaw)
    fx = c * f[0] + s * f[1]             # −yaw만큼 돌린 앞쪽 성분 (yaw = 0이면 f[0])
    fy = -s * f[0] + c * f[1]            # −yaw만큼 돌린 왼쪽 성분 (yaw = 0이면 f[1])
    pitch = ___  # ✍️ 옆에서 본 그림의 각
    roll = ___  # ✍️ 뒤에서 본 그림의 각 (부호 주의)
    thrust = ___  # ✍️ 힘 전체의 크기
    return thrust, np.array([roll, pitch, yaw])


check_attitude(acc_to_thrust_attitude)

64°나 누우면 추력의 위쪽 몫이 모자라서 높이를 못 지킨다. 그래서 다음 단계에서 기울기를 제한한다.

## 4. ③ 기울기 제한

자전거를 어느 이상 눕히지 않듯, 드론도 30°까지만 기울인다.
높이(a_z)는 지키고 **옆 가속도의 크기만** 같은 비율로 줄인다 (0단계의 "앞 두 칸만 줄이기").

- 위쪽으로 쓸 수 있는 가속도 `a_up = a_z + g` (급히 내려갈 때도 0.1g 아래로는 안 내린다)
- 30°에서 낼 수 있는 옆 가속도는 `h_max = a_up · tan(30°)`
- 지금 옆 가속도 `h = √(a_x² + a_y²)`가 `h_max`보다 크면 `acc[:2]`에 `h_max / h`를 곱한다

✍️ 마지막 두 줄을 채운다.

In [ ]:
def limit_tilt(acc, max_tilt):
    acc = np.array(acc, float)
    a_up = max(acc[2] + G, 0.1 * G)
    h_max = a_up * np.tan(max_tilt)
    h = np.hypot(acc[0], acc[1])
    if ___:  # ✍️
        acc[:2] *= ___  # ✍️
    return acc


check_tilt(limit_tilt, acc_to_thrust_attitude)

## 5. 첫 비행: 바람 속 호버

①②③이 다 모였으니 칩이 완성됐다. 칩에 넣고 날려 보자.
바람이 무게의 약 3%로 계속 밀고, 돌풍도 섞여 있다. (0, 0, 1)에 떠 있으면서 5초부터 위치 오차를 잰다.

아래 셀은 기본 게인과 **위치 I를 끈 것(PD만)** 을 비교한다.

In [ ]:
chip = MyChip(PositionController, acc_to_thrust_attitude, limit_tilt)

hover_in_wind(chip, 기본={}, PD만={"ki": 0})

**보고 생각하기**
- PD만이면 바람 쪽으로 밀린 채 선다. 바람이 주는 가속도 약 0.3 m/s²를 P가 버티려면 오차가 0.3 / 6 ≈ 5 cm 남아야 한다.
- I가 그 남은 오차를 지운다. 1회차의 "무거운 기체에서 처짐"과 같은 이야기다.

## 6. 실시간 비행: 날면서 바꿔 보기

내 칩을 단 드론을 **1배속으로** 계속 띄워 두고, 날고 있는 동안 슬라이더로 값을 바꾼다 (최대 3분, ■ 멈춤으로 끝).
바꾸기 전에 **먼저 맞혀 보고**, 그래프로 확인하자.

| 해 볼 것 | 먼저 맞혀 보기 |
|---|---|
| 목표 x를 0 → 1.0으로 확 옮기기 | 기울기가 몇 도까지 갈까? 4단계의 30° 제한에 걸릴까? |
| 바람을 0.03 N까지 올리기 | 드론은 바람이 불어오는 쪽으로 기울까, 불어 가는 쪽으로 기울까? |
| 바람을 둔 채 ki를 0으로 | 목표로 돌아올까, 밀린 채 머물까? (5단계의 PD만) |
| kd를 1로 낮추고 목표 x 옮기기 | 목표에서 딱 설까, 지나쳤다 돌아올까? |
| 원 그리기 켜기 | 목표(점선)를 제때 따라갈까? (8단계에서 이유를 본다) |

kd를 0까지 내리면 출렁임이 점점 커져서 날아가 버린다. 해 봤다면 바로 4로 되돌리자.

PlotJuggler에도 같은 값이 /pos/... 로 간다. 창은 아래 두 번째 셀로 연다 (Yes → OK). 3D 창은 `window=True`.

In [ ]:
live(chip)

In [ ]:
plotjuggler("week02")

## 7. ④ 웨이포인트: 링 코스

이제 지상국(노트북) 쪽 코드다. 내비게이션처럼 "경유지에 도착했습니다, 다음 안내를 시작합니다."
지금 목표에서 반경 15 cm 안에 들어오면 다음 점으로 넘어간다. 지상국은 1초에 50번 `update(지금 위치)`를 부르고,
돌려받은 점을 드론에 보낸다.

✍️ 도착 판정을 채운다. 다 돈 뒤(`self.done`)에는 판정하지 않는다. 그래야 `self.wps[self.i]`에서 오류가 안 난다.

In [ ]:
class WaypointFollower:
    def __init__(self, waypoints, radius=0.15):
        self.wps = np.asarray(waypoints, float)
        self.radius = radius
        self.i = 0                       # 지금 향하는 웨이포인트 번호

    @property
    def done(self):
        return self.i >= len(self.wps)

    def update(self, p):
        if not self.done and ___ < self.radius:  # ✍️ 지금 목표까지의 거리
            ___  # ✍️
        return self.wps[min(self.i, len(self.wps) - 1)]


ring_course(chip, WaypointFollower)

**보고 생각하기**
- 명령은 30°까지인데 최대 기울기가 30°를 조금 넘는다. 실제 자세가 명령을 따라가며 살짝 넘치기 때문이다.
- ④는 지상국에서, ①②③은 칩에서 돈다. 3~5회차의 대형, 합의, 회피도 지상국 쪽 코드다.
- `ring_course(chip, WaypointFollower, {"kd": 1})`처럼 브레이크를 약하게 하면 각 점에서 어떻게 될까? 6단계에서 본 것과 같은가?

## 8. 원 궤적과 앞먹임 (같이 보기)

6단계에서 원 그리기를 켜면 드론이 목표보다 늦게 따라왔다. 목표 위치만 줬기 때문이다.

원은 미리 다 알 수 있다. 원 위의 모든 점에서 속도(접선 방향)와 가속도(중심 방향)를 미리 안다.
굽은 길이 보이면 핸들을 미리 돌리듯, 궤적이 알려 주는 가속도를 ①의 마지막 항($\mathbf a_{ff}$)으로 더한다.
원 궤적은 칩에 올려 두고 칩이 직접 돌린다 (Crazyflie의 궤적 업로드와 같다).

아래 셀은 앞먹임을 켠 비행과 끈 비행을 비교한다.

In [ ]:
circle(chip)

**보고 생각하기**: 앞먹임이 없으면 원 바깥으로 밀리고 뒤처진다. 차가 바깥으로 밀린 다음에야 핸들을 돌리는 것과 같다.

## 9. 미션: 비행 시험 세 가지

기본 게인(kp 6, kd 4, ki 2)으로 네 칸 모두 초록이면 성공이다.

| 시험 | 기준 |
|---|---|
| 바람 속 호버 | 위치 오차 RMS < 5 cm |
| 링 코스 | 5/5 완주, 40초 안, 최대 기울기 ≤ 35° |
| 원 궤적 | RMS < 10 cm |

In [ ]:
flight_tests(chip, WaypointFollower)

빨리 끝났으면 슬라이더로 게인을 바꿔 가며 시험을 다시 날려 보자. 순서는 1회차와 같다: `kp`로 빠르기, `kd`로 출렁임, `ki`로 남는 오차.
네 칸을 다 초록으로 지키면서 링 코스를 가장 빨리 도는 게인은?

In [ ]:
tune(chip, WaypointFollower)

## 정리

| 오늘 쓴 것 | 한 줄 요약 |
|---|---|
| 기울기 | 기울여야 옆으로 간다. $a_x = g\tan\theta$ |
| ① 위치 제어기 | $\mathbf a = \text{PID}(\mathbf e_p) + K_d\mathbf e_v + \mathbf a_{ff}$ |
| ② 추력과 자세 | $\mathbf f = m(\mathbf a + g\hat z)$ 쪽으로 기체를 돌린다. 추력은 $|\mathbf f|$ |
| ③ 기울기 제한 | 30°를 넘으면 옆 가속도만 줄인다 |
| 실시간 비행 | 바람에는 기대어 버티고, 남는 오차는 I가, 출렁임은 D가 잡는다 |
| ④ 웨이포인트 | 반경 안에 들어오면 다음 점 |
| 앞먹임 | 미리 아는 가속도를 더하면 늦지 않는다 |

**나가기 전에 한 가지**: 드론이 바람에 밀리지 않고 제자리에 있을 때, 기체는 수평일까 기울어 있을까? 왜 그럴까?

다음 시간에는 오늘 만든 칩이 **여섯 대 모두**에서 돌고, 지상국이 여섯 대에게 대형을 만들라고 한다.

---
## 집에서 (선택, 25분): 진짜 칩은 참값을 모른다 · 칼만 필터

수업 시간에는 하지 않는다. 1~9단계가 끝났으면 집에서 이어서 해 보자. 3회차에는 필요 없다.

지금까지 칩은 시뮬레이터가 아는 **참값** 위치를 받았다. 실제 칩이 가진 것은 둘뿐이다.

| 센서 | 빠르기 | 약점 |
|---|---|---|
| IMU (가속도계) | 500 Hz | 바이어스 0.1 m/s². 적분하면 10초에 5 m 흘러간다 |
| 천장 카메라 | 100 Hz | 잡음 5 mm, 지연 20 ms |

눈을 감고 걸음 수로 위치를 짐작하면(IMU) 점점 틀어지고, 가끔 눈을 떠 확인하면(카메라) 바로잡힌다. **둘을 섞는다.**
고도 하나만 추정하는 1차원 칼만 필터를 만든다. 상태는 `x = [고도 z, 수직 속도 vz]`, 불확실성은 `P` (2×2).

- ⑤ **예측**: IMU 가속도로 dt 뒤를 예측한다. `x ← F·x + B·a`, `P ← F·P·Fᵀ + q·B·Bᵀ`
- ⑥ **칼만 이득**: 카메라를 얼마나 믿을지. `K = P·Hᵀ / (H·P·Hᵀ + r)` (카메라는 고도만 재므로 `H = [1, 0]`)
- ⑦ **보정**: `x ← x + K·(z_cam − H·x)`, `P ← (I − K·H)·P`

✍️ 다섯 줄을 채운다. 행렬 곱은 `@`, 바깥곱은 `np.outer`.

In [ ]:
class AltitudeKF:
    def __init__(self, z0=0.0, accel_noise=0.2, cam_noise=0.005):
        self.x = np.array([z0, 0.0])
        self.P = np.diag([1e-3, 1e-2])
        self.q = accel_noise ** 2          # 가속도계 잡음의 분산
        self.r = cam_noise ** 2            # 카메라 잡음의 분산

    @property
    def z(self):
        return float(self.x[0])

    def predict(self, a, dt):
        F = np.array([[1.0, dt], [0.0, 1.0]])
        B = np.array([0.5 * dt * dt, dt])
        self.x = ___  # ✍️
        self.P = ___  # ✍️

    def update(self, z_meas):
        H = np.array([1.0, 0.0])
        S = H @ self.P @ H + self.r
        K = ___  # ✍️
        self.x = ___  # ✍️
        self.P = ___  # ✍️


check_kf(AltitudeKF)

이제 고도를 무엇으로 재느냐만 바꿔서 1회차 고도 제어기를 날려 본다.

In [ ]:
kf_test(AltitudeKF)

**보고 생각하기**: 카메라 값을 그대로 쓰면 D 항이 5 mm 잡음을 미분해서 추력이 스텝마다 떨린다 (오른쪽 주황).
칼만 필터는 그 떨림을 수십 분의 1로 줄이고, 고도 오차도 카메라를 그대로 쓸 때보다 작다.
실제 기체에서는 이 떨림이 모터 소음과 열, 배터리 소모로 돌아온다.